# Kadi-APY Workshop: Tensile Test Data in Kadi4Mat

<a href="https://colab.research.google.com/github/msishuvo/thematic-workshop-kadiAPY/blob/main/notebooks/kadi_apy_workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this notebook we use kadi-apy to upload tensile test data to Kadi4Mat,
add metadata, and download the data again for plotting.

Each part has the same structure:

1. **Example**: complete code that we go through together.
2. **Reference**: the functions or commands used, with links to the kadi-apy documentation.
3. **Exercise**: a similar task for you. Only change the lines marked `TODO`.
   If you get stuck, open "Show solution" below the exercise.

How to work with the notebook:

- Run a cell with Shift + Enter, or with the run button on the left of the cell.
- Run the cells in order from top to bottom. Later cells use results of earlier ones.
- If something stops working, use Runtime > Restart session and run the cells again from the top.
- Questions: write in the meeting chat.

Before you start, use File > Save a copy in Drive and work in your copy.

## 0. Setup (before the workshop)

### 0.1 Store your access token as a Colab secret

The personal access token (PAT) allows kadi-apy to access Kadi4Mat on your
behalf. It is stored as a Colab secret so that it does not appear in the notebook.

1. Click the key icon ("Secrets") in the left sidebar.
2. Click "Add new secret". Name: `KADI_PAT`, value: your personal access token.
3. Switch on "Notebook access".

### 0.2 Install kadi-apy and download the workshop data

This takes about 30 seconds.

In [ ]:
%pip install -q kadi-apy

import os, subprocess, sys

REPO_DIR = "/content/thematic-workshop-kadiAPY"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", "https://github.com/msishuvo/thematic-workshop-kadiAPY.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.append(os.path.join(REPO_DIR, "exercises"))

print("Setup complete. Data files:", sorted(os.listdir("data/tensile_tests")))

### 0.3 Connect to Kadi4Mat

This cell writes the Kadi4Mat address and your token to the kadi-apy
configuration file and checks the connection. The output should start with
"Connected to". On your own computer, the same configuration file is created with
the commands `kadi-apy config create`, `kadi-apy config set-host` and
`kadi-apy config set-pat` (see [Configuration](https://kadi-apy.readthedocs.io/en/stable/setup/configuration.html)).

In [ ]:
import configparser
from pathlib import Path
from google.colab import userdata

host = "https://kadi4mat.iam.kit.edu"

try:
    pat = userdata.get("KADI_PAT").strip()
except Exception as error:
    raise RuntimeError(
        "Could not read the secret KADI_PAT. Check step 0.1: is the name "
        "spelled exactly like this, and is 'Notebook access' switched on?"
    ) from error

config = configparser.ConfigParser()
config["global"] = {"default": "workshop"}
config["workshop"] = {"host": host, "pat": pat}
config_path = Path.home() / ".kadiconfig"
with open(config_path, "w") as f:
    config.write(f)
config_path.chmod(0o600)

from kadi_apy import KadiManager

manager = KadiManager()
me = manager.pat_user.meta
print(f"Connected to {host} as {me.get('displayname', me.get('id'))}")

### 0.4 Set your prefix

Identifiers in Kadi4Mat must be unique, so everything you create starts with
your initials. Use lowercase letters only, e.g. `jd` for Jane Doe.

In [ ]:
PREFIX = "xy"  # TODO: change to your initials

import re
assert re.fullmatch(r"[a-z0-9]{2,10}", PREFIX), "Use 2-10 lowercase letters or digits."
assert PREFIX != "xy", "Please change 'xy' to your own initials."
CAMPAIGN_TAG = f"{PREFIX}-tensile-campaign"
print("Prefix set to", PREFIX)

## 1. Using the kadi-apy documentation (5 min)

The documentation is at <https://kadi-apy.readthedocs.io/en/stable/>. For this workshop, two pages are relevant:

| Page | Content |
|---|---|
| [Usage > Library](https://kadi-apy.readthedocs.io/en/stable/usage/lib.html) | The Python functions. Start with `KadiManager`, then `Record`, `Collection` and `Search`. |
| [Usage > CLI](https://kadi-apy.readthedocs.io/en/stable/usage/cli.html) | All command line commands with their options. |

There are also [example scripts](https://gitlab.com/iam-cms/kadi-apy/-/tree/master/examples)
in the kadi-apy source repository.

**Reading a function entry.** On the Library page, the entry for `upload_file`
(in the section `Record`) looks like this:

```
upload_file(file_path, file_name=None, file_description=None, force=False)
    Upload a file to a record.
    Parameters: file_path (str) - The path to the file (incl. name of the file).
                file_name (str, optional) - The name under which the file should be stored ...
                ...
    Returns:    The final response object of the upload process.
```

- Parameters without `=` are required (`file_path`). Parameters with `=` are optional
  and the value after `=` is the default.
- Most functions return a *response object*. `response.ok` tells you whether the request
  was successful, and `response.json()` returns the data sent back by Kadi4Mat.
- The parameters of `manager.record(...)` (`id`, `identifier`, `title`, `create`, ...)
  are described at the top of the `Record` section.

**Help inside the notebook.** The same information is available without leaving Colab:
`help(...)` for Python functions and `--help` for commands. In the `help` output,
`self` stands for the record itself and is not passed when calling the function.

In [ ]:
from kadi_apy.lib.resources.records import Record

help(Record.upload_file)

In [ ]:
!kadi-apy records --help

In [ ]:
!kadi-apy records add-files --help

## 2. Command line interface (15 min)

Most actions in the Kadi4Mat web interface can also be done with kadi-apy
commands. In Colab, lines starting with `!` are run as terminal commands, and
`{PREFIX}` is replaced with the value of the Python variable.

Keep Kadi4Mat open in a second browser tab and check the result after each step.

### 2.1 Example: record for sample S01

Create the record. `-i` is the identifier, `-t` the title:

In [ ]:
!kadi-apy records create -i {PREFIX}-sample-s01 -t "Steel sample S01"

Upload the measurement file. `-R` selects the record by its identifier, `-n` is the file:

In [ ]:
!kadi-apy records add-files -R {PREFIX}-sample-s01 -n data/tensile_tests/S01_tensile.csv

Add metadata. Each entry has a name (`-m`), a value (`-v`), a type (`-t`, default:
string) and optionally a unit (`-u`):

In [ ]:
!kadi-apy records add-metadatum -R {PREFIX}-sample-s01 -m material -v "S355 steel"
!kadi-apy records add-metadatum -R {PREFIX}-sample-s01 -m specimen_length -v 80 -t float -u mm
!kadi-apy records add-metadatum -R {PREFIX}-sample-s01 -m test_temperature -v 23 -t float -u degC

Add a tag and show the result (`-m` shows the metadata, `-l` the file list):

In [ ]:
!kadi-apy records add-tag -R {PREFIX}-sample-s01 -t {CAMPAIGN_TAG}
!kadi-apy records show-info -R {PREFIX}-sample-s01 -m -l

### 2.2 Reference

| Command | Purpose |
|---|---|
| [`kadi-apy records create`](https://kadi-apy.readthedocs.io/en/stable/usage/cli.html#kadi-apy-records-create) | Create a record (or get it, if the identifier exists) |
| [`kadi-apy records add-files`](https://kadi-apy.readthedocs.io/en/stable/usage/cli.html#kadi-apy-records-add-files) | Upload a file or all files of a folder |
| [`kadi-apy records add-metadatum`](https://kadi-apy.readthedocs.io/en/stable/usage/cli.html#kadi-apy-records-add-metadatum) | Add one metadata entry |
| [`kadi-apy records add-tag`](https://kadi-apy.readthedocs.io/en/stable/usage/cli.html#kadi-apy-records-add-tag) | Add a tag |
| [`kadi-apy records show-info`](https://kadi-apy.readthedocs.io/en/stable/usage/cli.html#kadi-apy-records-show-info) | Show the record, its metadata (`-m`) and files (`-l`) |

Related commands: [`kadi-apy records add-metadata`](https://kadi-apy.readthedocs.io/en/stable/usage/cli.html#kadi-apy-records-add-metadata) (several entries from a JSON file),
[`kadi-apy records get-file`](https://kadi-apy.readthedocs.io/en/stable/usage/cli.html#kadi-apy-records-get-file) (download files), [`kadi-apy records edit`](https://kadi-apy.readthedocs.io/en/stable/usage/cli.html#kadi-apy-records-edit) (change title,
description, ...), [`kadi-apy records delete`](https://kadi-apy.readthedocs.io/en/stable/usage/cli.html#kadi-apy-records-delete). Add `--help` to any command to see its options.

### 2.3 Exercise: record for the testing machine

Create a record for the tensile testing machine:

- identifier `{PREFIX}-tensile-machine`, title "Tensile testing machine"
- metadata `max_force` = 100 (type float, unit kN)
- metadata `location` = "Lab 1" (text)
- show the record with its metadata

We will link the samples to this record later.

In [ ]:
# TODO 1: create the record

# TODO 2: add the metadatum max_force

# TODO 3: add the metadatum location

# TODO 4: show the record with its metadata

<details><summary>Show solution</summary>

```python
!kadi-apy records create -i {PREFIX}-tensile-machine -t "Tensile testing machine"
!kadi-apy records add-metadatum -R {PREFIX}-tensile-machine -m max_force -v 100 -t float -u kN
!kadi-apy records add-metadatum -R {PREFIX}-tensile-machine -m location -v "Lab 1"
!kadi-apy records show-info -R {PREFIX}-tensile-machine -m
```

</details>

## 3. Python (20 min)

The same steps in Python. Python is useful as soon as you need loops,
calculations or have to read your own files.

### 3.1 Example: record for sample S02

- `manager` is the connection to Kadi4Mat, created in step 0.3.
- `manager.record(identifier=..., title=..., create=True)` returns the record
  with this identifier, or creates it if it does not exist yet. Running the
  cell twice does not create a duplicate.
- Metadata is a list of dictionaries with the fields `key`, `type`, `value`
  and optionally `unit`. Types: `str`, `int`, `float`, `bool`.
- `force=True` overwrites entries or files that already exist.

In [ ]:
record = manager.record(
    identifier=f"{PREFIX}-sample-s02",
    title="Steel sample S02",
    create=True,
)
print("Record:", record)

metadata = [
    {"key": "material", "type": "str", "value": "S355 steel"},
    {"key": "specimen_length", "type": "float", "value": 80.0, "unit": "mm"},
    {"key": "test_temperature", "type": "float", "value": 23.0, "unit": "degC"},
]
record.add_metadata(metadata, force=True)

response = record.upload_file("data/tensile_tests/S02_tensile.csv", force=True)
print("Upload successful:", response.ok)

record.add_tag(CAMPAIGN_TAG)

# Read a value back from Kadi4Mat
print(record.get_metadatum("specimen_length"))

### 3.2 Reference

| Function | Purpose |
|---|---|
| [`KadiManager.record`](https://kadi-apy.readthedocs.io/en/stable/usage/lib.html#kadi_apy.lib.core.KadiManager.record) | Get or create a record. Parameters: see [`Record`](https://kadi-apy.readthedocs.io/en/stable/usage/lib.html#kadi_apy.lib.resources.records.Record) |
| [`Record.add_metadata`](https://kadi-apy.readthedocs.io/en/stable/usage/lib.html#kadi_apy.lib.resources.records.Record.add_metadata) | Add a list of metadata entries |
| [`Record.upload_file`](https://kadi-apy.readthedocs.io/en/stable/usage/lib.html#kadi_apy.lib.resources.records.Record.upload_file) | Upload a file |
| [`Record.add_tag`](https://kadi-apy.readthedocs.io/en/stable/usage/lib.html#kadi_apy.lib.resources.records.Record.add_tag) | Add a tag |
| [`Record.get_metadatum`](https://kadi-apy.readthedocs.io/en/stable/usage/lib.html#kadi_apy.lib.resources.records.Record.get_metadatum) | Read one metadata entry |

Related functions: [`Record.add_metadatum`](https://kadi-apy.readthedocs.io/en/stable/usage/lib.html#kadi_apy.lib.resources.records.Record.add_metadatum) (one entry),
[`Record.remove_metadatum`](https://kadi-apy.readthedocs.io/en/stable/usage/lib.html#kadi_apy.lib.resources.records.Record.remove_metadatum), [`Record.upload_string_to_file`](https://kadi-apy.readthedocs.io/en/stable/usage/lib.html#kadi_apy.lib.resources.records.Record.upload_string_to_file)
(store text as a file, e.g. notes), [`Record.remove_tag`](https://kadi-apy.readthedocs.io/en/stable/usage/lib.html#kadi_apy.lib.resources.records.Record.remove_tag), [`Record.edit`](https://kadi-apy.readthedocs.io/en/stable/usage/lib.html#kadi_apy.lib.resources.records.Record.edit)
(change title, description, ...). In Colab, `record.upload_file?` also shows the documentation.

### 3.3 Exercise: record for sample S03

Do the same for sample S03: create the record `{PREFIX}-sample-s03` with the title
"Steel sample S03", add the three metadata entries, upload
`data/tensile_tests/S03_tensile.csv` and add the tag `CAMPAIGN_TAG`.

In [ ]:
# TODO 1: get or create the record
record = ...

# TODO 2: create the metadata list (material, specimen_length, test_temperature)
metadata = [
]
record.add_metadata(metadata, force=True)

# TODO 3: upload the file

# TODO 4: add the tag

print("Done. Check the record in Kadi4Mat:", record)

<details><summary>Show solution</summary>

```python
record = manager.record(
    identifier=f"{PREFIX}-sample-s03",
    title="Steel sample S03",
    create=True,
)

metadata = [
    {"key": "material", "type": "str", "value": "S355 steel"},
    {"key": "specimen_length", "type": "float", "value": 80.0, "unit": "mm"},
    {"key": "test_temperature", "type": "float", "value": 23.0, "unit": "degC"},
]
record.add_metadata(metadata, force=True)

record.upload_file("data/tensile_tests/S03_tensile.csv", force=True)

record.add_tag(CAMPAIGN_TAG)

print("Done. Check the record in Kadi4Mat:", record)
```

</details>

## 4. Batch upload of a test campaign (20 min)

A typical situation in the lab: the testing machine has written one result
file per specimen into a folder. Instead of uploading each file by hand and
copying values into the metadata, a loop does this for every file. In addition,
the samples are grouped in a collection and linked to the machine record.

### 4.1 Example: collections and record links

Sample S01 is added to a new collection for the campaign and linked to the
machine record from exercise 2.3.

The Python functions `add_collection_link` and `link_record` expect the numeric
ID of the collection or record. If you only know the identifier, get the record
or collection with `manager.record(identifier=...)` or
`manager.collection(identifier=...)` first and use its `.id`.

In [ ]:
# Collection for the whole campaign (created if it does not exist yet)
collection = manager.collection(
    identifier=f"{PREFIX}-tensile-campaign-2026",
    title=f"Tensile test campaign 2026 ({PREFIX})",
    create=True,
)

# The records from part 2, looked up by their identifiers.
# (create=True only creates them if they do not exist yet, e.g. if 2.3 was skipped)
sample = manager.record(identifier=f"{PREFIX}-sample-s01",
                        title="Steel sample S01", create=True)
machine = manager.record(identifier=f"{PREFIX}-tensile-machine",
                         title="Tensile testing machine", create=True)
print("Collection ID:", collection.id, " Sample ID:", sample.id, " Machine ID:", machine.id)

# Add the sample to the collection
response = sample.add_collection_link(collection.id)
print("Added to collection:", response.ok)

# Link the sample to the machine. The name describes the relation.
response = sample.link_record(machine.id, name="tested with")
print("Linked to machine:", response.ok)

Check the result in Kadi4Mat: the collection contains sample S01, and the record
S01 has the link "tested with" to the machine. Run this cell only once, because
`link_record` tries to create the link again each time it runs.

With the command line, the identifiers can be used directly
(`-C` collection identifier, `-L` identifier of the record to link):

```
!kadi-apy records add-collection-link -R {PREFIX}-sample-s01 -C {PREFIX}-tensile-campaign-2026
!kadi-apy records add-record-link -R {PREFIX}-sample-s01 -L {PREFIX}-tensile-machine -n "tested with"
```

### 4.2 Example: evaluating all files in a folder

The function `evaluate` (in `exercises/tensile.py`) reads one file and
calculates the characteristic values. For your own instrument, this is the
part you would adapt.

In [ ]:
from tensile import evaluate, read_curve

evaluate("data/tensile_tests/S01_tensile.csv")

A `for` loop runs the indented block once for every element, here for every CSV
file in the folder:

In [ ]:
from pathlib import Path

for csv_file in sorted(Path("data/tensile_tests").glob("*.csv")):
    sample_id = csv_file.stem.split("_")[0]  # "S01_tensile" -> "S01"
    results = evaluate(csv_file)
    print(sample_id, results)

### 4.3 Reference

| Function | Purpose |
|---|---|
| [`KadiManager.collection`](https://kadi-apy.readthedocs.io/en/stable/usage/lib.html#kadi_apy.lib.core.KadiManager.collection) | Get or create a collection. Parameters: see [`Collection`](https://kadi-apy.readthedocs.io/en/stable/usage/lib.html#kadi_apy.lib.resources.collections.Collection) |
| [`Record.add_collection_link`](https://kadi-apy.readthedocs.io/en/stable/usage/lib.html#kadi_apy.lib.resources.records.Record.add_collection_link) | Add a record to a collection (collection ID) |
| [`Record.link_record`](https://kadi-apy.readthedocs.io/en/stable/usage/lib.html#kadi_apy.lib.resources.records.Record.link_record) | Link two records (ID of the target record, name of the link) |

Related functions: [`Record.remove_collection_link`](https://kadi-apy.readthedocs.io/en/stable/usage/lib.html#kadi_apy.lib.resources.records.Record.remove_collection_link), [`Record.get_record_links`](https://kadi-apy.readthedocs.io/en/stable/usage/lib.html#kadi_apy.lib.resources.records.Record.get_record_links),
[`Record.delete_record_link`](https://kadi-apy.readthedocs.io/en/stable/usage/lib.html#kadi_apy.lib.resources.records.Record.delete_record_link).

Command line (identifiers or IDs): [`kadi-apy records add-collection-link`](https://kadi-apy.readthedocs.io/en/stable/usage/cli.html#kadi-apy-records-add-collection-link),
[`kadi-apy records add-record-link`](https://kadi-apy.readthedocs.io/en/stable/usage/cli.html#kadi-apy-records-add-record-link), [`kadi-apy collections add-record-link`](https://kadi-apy.readthedocs.io/en/stable/usage/cli.html#kadi-apy-collections-add-record-link).

### 4.4 Exercise: upload all samples

Combine the examples from part 3, 4.1 and 4.2. For every file, the code below
creates the record and the metadata list. Fill in the five `TODO` lines: metadata,
file, tag, collection and the link to the machine.

In [ ]:
UNITS = {"youngs_modulus": "GPa", "yield_strength": "MPa",
         "tensile_strength": "MPa", "elongation_at_break": "%"}

for csv_file in sorted(Path("data/tensile_tests").glob("*.csv")):
    sample_id = csv_file.stem.split("_")[0]

    record = manager.record(
        identifier=f"{PREFIX}-sample-{sample_id.lower()}",
        title=f"Steel sample {sample_id}",
        create=True,
    )

    results = evaluate(csv_file)
    metadata = [
        {"key": "material", "type": "str", "value": "S355 steel"},
        {"key": "test_standard", "type": "str", "value": "ISO 6892-1"},
    ]
    for key, value in results.items():
        metadata.append({"key": key, "type": "float", "value": value, "unit": UNITS[key]})

    # TODO 1: add the metadata to the record (use force=True)

    # TODO 2: upload the CSV file (upload_file expects a string: str(csv_file))

    # TODO 3: add the tag CAMPAIGN_TAG

    # TODO 4: add the record to the collection (collection from 4.1)

    # TODO 5: link the record to the machine (machine from 4.1, name "tested with").
    #         S01 is already linked in 4.1, so skip it: if sample_id != "S01":

    print(sample_id, results)

print("Done. Open the collection in Kadi4Mat.")

<details><summary>Show solution</summary>

```python
    record.add_metadata(metadata, force=True)
    record.upload_file(str(csv_file), force=True)
    record.add_tag(CAMPAIGN_TAG)
    record.add_collection_link(collection.id)
    if sample_id != "S01":
        record.link_record(machine.id, name="tested with")
```

</details>

## 5. Search, download and plot (10 min)

### 5.1 Example: search by tag

`search_resources` returns a response object. `.json()["items"]` contains the
records that were found, as dictionaries.

In [ ]:
response = manager.search.search_resources("record", tag=[CAMPAIGN_TAG], per_page=100)
found = response.json()["items"]
print(f"Found {len(found)} records with tag '{CAMPAIGN_TAG}'\n")

print(f"{'Record':<22} {'Yield [MPa]':>12} {'UTS [MPa]':>10}")
for item in found:
    record = manager.record(id=item["id"])
    extras = {e["key"]: e.get("value") for e in record.meta["extras"]}
    print(f"{record.meta['title']:<22} {extras.get('yield_strength', '-'):>12} "
          f"{extras.get('tensile_strength', '-'):>10}")

### 5.2 Reference

| Function | Purpose |
|---|---|
| [`Search.search_resources`](https://kadi-apy.readthedocs.io/en/stable/usage/lib.html#kadi_apy.lib.search.Search.search_resources) | Search records, collections, ... |
| [`Record.get_filelist`](https://kadi-apy.readthedocs.io/en/stable/usage/lib.html#kadi_apy.lib.resources.records.Record.get_filelist) | List the files of a record |
| [`Record.download_file`](https://kadi-apy.readthedocs.io/en/stable/usage/lib.html#kadi_apy.lib.resources.records.Record.download_file) | Download one file |

Related functions: [`Record.download_all_files`](https://kadi-apy.readthedocs.io/en/stable/usage/lib.html#kadi_apy.lib.resources.records.Record.download_all_files) (all files as ZIP archive),
[`Record.export`](https://kadi-apy.readthedocs.io/en/stable/usage/lib.html#kadi_apy.lib.resources.records.Record.export) (export the record, e.g. as JSON or PDF). The search options
(`tag`, `q`, `collection`, ...) are the same as for [`kadi-apy records get-records`](https://kadi-apy.readthedocs.io/en/stable/usage/cli.html#kadi-apy-records-get-records);
see `!kadi-apy records get-records --help`.

### 5.3 Exercise: download and plot

For every record found, download its CSV file and plot the curve.

In [ ]:
import matplotlib.pyplot as plt

os.makedirs("downloads", exist_ok=True)

for item in found:
    record = manager.record(id=item["id"])
    for file_info in record.get_filelist().json()["items"]:
        if file_info["name"].endswith(".csv"):
            local_path = f"downloads/{file_info['name']}"

            # TODO 1: download the file to local_path (file ID: file_info["id"])

            strain, stress = read_curve(local_path)
            plt.plot(strain, stress, label=record.meta["title"])

# TODO 2: label the axes "Strain [%]" and "Stress [MPa]"  (plt.xlabel(...), plt.ylabel(...))

plt.title("Tensile test campaign")
plt.legend()
plt.show()

<details><summary>Show solution</summary>

```python
            record.download_file(file_info["id"], local_path)

plt.xlabel("Strain [%]")
plt.ylabel("Stress [MPa]")
```

</details>

## Next steps

To try this with your own data, upload a few files with the folder icon in the
left sidebar, change the folder in part 4 and adapt `evaluate`, or leave it out
and upload the files with basic metadata only.

Note: Colab runs on Google servers. Check your institute's data policy before
using unpublished lab data here. kadi-apy can also be installed on your own
computer with `pip install kadi-apy`.

Links:
- kadi-apy documentation: https://kadi-apy.readthedocs.io/en/stable/
- Functions and commands used in this workshop: https://github.com/msishuvo/thematic-workshop-kadiAPY/blob/main/docs/kadi-apy-reference.md
- kadi-apy source code and examples: https://gitlab.com/iam-cms/kadi-apy
- Workshop material: https://github.com/msishuvo/thematic-workshop-kadiAPY